# 再帰型ニューラルネットワーク

系列データでは、現在の入力だけでは答えが決まらないことがあります。RNN は、前の状態 h_{t-1} と現在の入力 x_t から次の状態 h_t を作り、読んできた情報を小さなメモとして運びます。LSTM と GRU は、このメモに何を残し、何を忘れ、何を出力するかをゲートで制御します。

## 単純 RNN の状態更新

単純 RNN は h_t = tanh(W_x x_t + W_h h_{t-1} + b) で状態を更新します。W_x は新しい入力の効き方、W_h は過去の状態の残り方を決めます。まず 1 次元の系列で、状態が時刻ごとにどう変わるかを追います。

In [ ]:
import math
import random
from statistics import mean


def sigmoid(x):
    if x >= 0:
        z = math.exp(-x)
        return 1 / (1 + z)
    z = math.exp(x)
    return z / (1 + z)


def simple_rnn_forward(xs, wx=1.0, wh=0.72, b=0.0):
    h = 0.0
    trace = []
    for t, x in enumerate(xs):
        pre = wx * x + wh * h + b
        h = math.tanh(pre)
        trace.append({'t': t, 'x': x, 'pre': pre, 'h': h})
    return h, trace

seq = [0.2, -0.1, 0.5, 0.3, -0.4, 0.1]
final_h, trace = simple_rnn_forward(seq)
for row in trace:
    print('t=', row['t'], 'x=', round(row['x'], 3), 'pre=', round(row['pre'], 4), 'h=', round(row['h'], 4))
print('final h:', round(final_h, 4))

## 長い系列で勾配が壊れやすい理由

時刻をまたいで学習信号を戻すとき、勾配は W_h と tanh の導関数を何度も掛けます。係数が 1 より小さい状態が続けば急速に小さくなり、1 より大きい状態が続けば急速に大きくなります。これが勾配消失と勾配爆発です。

In [ ]:
def tanh_derivative_from_h(h):
    return 1.0 - h * h


def backprop_chain_gain(xs, wx=1.0, wh=0.72):
    _, trace = simple_rnn_forward(xs, wx=wx, wh=wh)
    gains = []
    gain = 1.0
    for row in reversed(trace):
        gain *= abs(wh * tanh_derivative_from_h(row['h']))
        gains.append(gain)
    return list(reversed(gains))

long_seq = [0.4] + [0.03 * math.sin(i) for i in range(1, 41)]
for wh in [0.45, 0.72, 1.05]:
    gains = backprop_chain_gain(long_seq, wh=wh)
    print('wh=', wh, 'first-step gain=', f'{gains[0]:.8f}', 'middle=', f'{gains[20]:.8f}', 'last=', f'{gains[-1]:.8f}')

単純 RNN は、状態を毎時刻まるごと上書きします。短い依存なら機能しますが、遠い過去の情報を残すには不安定です。ゲート付きモデルは、状態をどれくらい保持するかを明示的に持たせます。

## LSTM はセル状態を足し算で運ぶ

LSTM には h とは別に c というセル状態があります。forget gate は古い c を残す割合、input gate は新しい候補を入れる割合、output gate は外へ見せる割合を決めます。c_t = f_t c_{t-1} + i_t g_t という足し算の道を持つため、情報を長く残しやすくなります。

In [ ]:
def lstm_step(x, h, c, p):
    z = p['wx'] * x + p['wh'] * h
    i = sigmoid(z + p['bi'])
    f = sigmoid(z + p['bf'])
    g = math.tanh(z + p['bg'])
    o = sigmoid(z + p['bo'])
    c_new = f * c + i * g
    h_new = o * math.tanh(c_new)
    return h_new, c_new, {'i': i, 'f': f, 'g': g, 'o': o}

params_lstm = {'wx': 1.1, 'wh': 0.25, 'bi': -0.4, 'bf': 1.7, 'bg': 0.0, 'bo': 0.8}
h = 0.0
c = 0.0
for t, x in enumerate(seq):
    h, c, gates = lstm_step(x, h, c, params_lstm)
    print('t=', t, 'x=', round(x, 2), 'i/f/o=', [round(gates[k], 3) for k in ['i', 'f', 'o']], 'c=', round(c, 4), 'h=', round(h, 4))

forget gate が 1 に近いと古いセル状態が残り、0 に近いと消えます。input gate が小さいと新しい入力に状態を乱されにくくなります。LSTM は、記憶を保持する経路と出力する経路を分けている点が単純 RNN と大きく違います。

## GRU は状態を一つにまとめる

GRU は LSTM より軽い設計です。update gate は過去状態を残す割合、reset gate は候補状態を作るときに過去をどれだけ使うかを決めます。セル状態 c を別に持たず、h だけで保持と更新を行います。

In [ ]:
def gru_step(x, h, p):
    z = sigmoid(p['wz_x'] * x + p['wz_h'] * h + p['bz'])
    r = sigmoid(p['wr_x'] * x + p['wr_h'] * h + p['br'])
    h_candidate = math.tanh(p['wh_x'] * x + p['wh_h'] * (r * h) + p['bh'])
    h_new = z * h + (1 - z) * h_candidate
    return h_new, {'z': z, 'r': r, 'candidate': h_candidate}

params_gru = {'wz_x': -0.4, 'wz_h': 0.8, 'bz': 1.2, 'wr_x': 0.5, 'wr_h': 0.4, 'br': 0.0, 'wh_x': 1.0, 'wh_h': 0.65, 'bh': 0.0}
h = 0.0
for t, x in enumerate(seq):
    h, gates = gru_step(x, h, params_gru)
    print('t=', t, 'x=', round(x, 2), 'z/r=', round(gates['z'], 3), round(gates['r'], 3), 'candidate=', round(gates['candidate'], 4), 'h=', round(h, 4))

## パラメータ数で設計差を見る

LSTM は input、forget、candidate、output の 4 系統を持つため、同じ hidden size なら RNN より重くなります。GRU は 3 系統です。軽さ、記憶力、学習安定性の釣り合いで選びます。

In [ ]:
def param_count(input_size, hidden_size, gates, two_bias=False):
    bias_terms = 2 * hidden_size if two_bias else hidden_size
    return gates * (hidden_size * input_size + hidden_size * hidden_size + bias_terms)

input_size = 16
hidden_size = 64
for name, gates in [('RNN', 1), ('GRU', 3), ('LSTM', 4)]:
    print(name, 'single-bias=', param_count(input_size, hidden_size, gates), 'two-bias=', param_count(input_size, hidden_size, gates, two_bias=True))

## 遠い過去を覚える小さな課題

系列の最初の値が正なら 1、負なら 0 と判定する課題を作ります。途中の値はノイズです。最後の状態だけで判定するなら、最初の情報を最後まで残せるかが勝負になります。

In [ ]:
def make_memory_data(n=400, seq_len=35, seed=0):
    rng = random.Random(seed)
    samples = []
    for _ in range(n):
        first = rng.choice([-1.0, 1.0])
        xs = [first] + [rng.gauss(0.0, 0.35) for _ in range(seq_len - 1)]
        y = 1 if first > 0 else 0
        samples.append((xs, y))
    return samples


def classify_from_state(h):
    return 1 if h >= 0 else 0


def eval_plain_rnn(samples, wh):
    correct = 0
    abs_states = []
    for xs, y in samples:
        h, _ = simple_rnn_forward(xs, wx=0.85, wh=wh)
        correct += int(classify_from_state(h) == y)
        abs_states.append(abs(h))
    return correct / len(samples), mean(abs_states)

samples = make_memory_data()
for wh in [0.3, 0.6, 0.9]:
    acc, mag = eval_plain_rnn(samples, wh)
    print('plain RNN wh=', wh, 'accuracy=', round(acc, 3), 'avg |h_T|=', round(mag, 4))

単純 RNN では、wh を小さくすると最初の情報が消えやすくなります。wh を大きくすれば保持は強くなりますが、飽和や勾配爆発の危険も増えます。ゲートは、この保持の強さを入力や状態に応じて変えられるようにします。

In [ ]:
def controlled_lstm_forward(xs):
    c = 0.0
    h = 0.0
    for t, x in enumerate(xs):
        if t == 0:
            i = 0.98
            f = 0.00
        else:
            i = 0.02
            f = 0.97
        g = math.tanh(1.8 * x)
        o = 0.95
        c = f * c + i * g
        h = o * math.tanh(c)
    return h, c


def controlled_gru_forward(xs):
    h = 0.0
    for t, x in enumerate(xs):
        if t == 0:
            z = 0.02
        else:
            z = 0.97
        candidate = math.tanh(1.8 * x + 0.25 * h)
        h = z * h + (1 - z) * candidate
    return h

correct_lstm = 0
correct_gru = 0
for xs, y in samples:
    h_lstm, c_lstm = controlled_lstm_forward(xs)
    h_gru = controlled_gru_forward(xs)
    correct_lstm += int(classify_from_state(c_lstm) == y)
    correct_gru += int(classify_from_state(h_gru) == y)
print('controlled LSTM accuracy:', round(correct_lstm / len(samples), 3))
print('controlled GRU accuracy :', round(correct_gru / len(samples), 3))

この比較は学習済みモデルの性能競争ではありません。最初の時刻では入力を強く書き込み、それ以降は保持を強める制御を置くと、ゲートが遠い情報を残すための仕組みだと分かります。実務では、このようなゲート値をデータから学習します。

RNN は、系列を左から右へ読みながら状態を更新します。単純 RNN は構造が軽い一方、長い依存で勾配が消えやすくなります。LSTM はセル状態と 3 種のゲートで記憶を交通整理し、GRU はより少ないゲートで保持と更新をまとめます。モデル選択では、系列長、必要な記憶、計算量、並列化しやすさを分けて判断します。